In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## ------------------------------------------------------------------
## STEP 1: Get Data
## ------------------------------------------------------------------
# Download 5 years of SPY data
data = yf.download('SPY', start='2025-01-01', end='2025-10-01')

## ------------------------------------------------------------------
## STEP 2: Feature Engineering (Our "X" variables)
## ------------------------------------------------------------------
# 1. Create a 10-day Rate of Change (ROC)
data['ROC'] = data['Close'].pct_change(10)

# 2. Create a 14-day Relative Strength Index (RSI)
delta = data['Close'].diff(1)
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.rolling(window=14, min_periods=1).mean()
avg_loss = loss.rolling(window=14, min_periods=1).mean()
rs = avg_gain / avg_loss
data['RSI'] = 100 - (100 / (1 + rs))

## ------------------------------------------------------------------
## STEP 3: Create Labels (Our "Y" variable)
## ------------------------------------------------------------------
# This is our "target" question: 
# "What was the % return 5 days in the future?"
future_return = data['Close'].pct_change(5).shift(-5)

# Threshold signal 
UPPER_THRESHOLD = 0.02  # We want a 2% gain to signal "Buy"
LOWER_THRESHOLD = -0.02 # We want a -2% loss to signal "Sell"

# 1. Create a list of our conditions
conditions = [
    (future_return > UPPER_THRESHOLD),   # Condition for Buy (1)
    (future_return < LOWER_THRESHOLD)    # Condition for Sell (-1)
]

# 2. Create a list of our choices (labels) for each condition
choices = [
    1,   # Label for "Buy"
    -1   # Label for "Sell"
]

# Create the label: 
# If the future return was > 2%, label it '1' (Buy)
# Otherwise, label it '0' (Hold/Sell)
data['Target'] = np.select(conditions, choices, default=0)

## ------------------------------------------------------------------
## STEP 4: Prepare Data for the Model
## ------------------------------------------------------------------
# Drop any rows with missing data (from the rolling windows)
data = data.dropna()

# Create our X (features) and Y (target)
X = data[['ROC', 'RSI']]
Y = data['Target']

# IMPORTANT: Split data into a training set and a test set
# We must do this to see if the model learned a real pattern
# or just "memorized" the past.
split_percentage = 0.8
split_index = int(len(data) * split_percentage)

# Train on the first 80% of the data
X_train = X[:split_index]
Y_train = Y[:split_index]

# Test on the last 20% of the data
X_test = X[split_index:]
Y_test = Y[split_index:]

print(f"Training on {len(X_train)} data points.")
print(f"Testing on {len(X_test)} data points.")

## ------------------------------------------------------------------
## STEP 5: Create and Train the Model
## ------------------------------------------------------------------
# Create the Random Forest model
# n_estimators=100 means we want 100 trees in our forest
model = RandomForestClassifier(n_estimators=500, random_state=42, max_depth=5)

# Train the model on the training data
model.fit(X_train, Y_train)

## ------------------------------------------------------------------
## STEP 6: Make Predictions on the Test Data
## ------------------------------------------------------------------
# Ask the trained model to predict the labels for the "unseen" X_test data
predictions = model.predict(X_test)

## ------------------------------------------------------------------
## STEP 7: Show the Results
## ------------------------------------------------------------------
print("\n--- Model Evaluation (On Unseen Test Data) ---")
print(f"Model Accuracy: {accuracy_score(Y_test, predictions) * 100:.2f}%")

# Create a small DataFrame to easily compare the last 20 predictions
results = pd.DataFrame({
    'Actual_Signal': Y_test,
    'Model_Prediction': predictions
})

print("\n--- Last 20 Predictions vs. Actual ---")
print(results.tail(70))

/tmp/ipykernel_563/35785194.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('SPY', start='2025-01-01', end='2025-10-01')
[*********************100%***********************]  1 of 1 completed


Training on 140 data points.
Testing on 36 data points.

--- Model Evaluation (On Unseen Test Data) ---
Model Accuracy: 91.67%

--- Last 20 Predictions vs. Actual ---
            Actual_Signal  Model_Prediction
Date                                       
2025-08-11              0                 0
2025-08-12              0                 0
2025-08-13              0                 0
2025-08-14              0                 0
2025-08-15              0                 0
2025-08-18              0                 0
2025-08-19              0                 0
2025-08-20              0                 0
2025-08-21              1                 0
2025-08-22              0                 0
2025-08-25              0                 0
2025-08-26              0                 0
2025-08-27              0                 0
2025-08-28              0                 0
2025-08-29              0                 0
2025-09-02              0                -1
2025-09-03              0                

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE  

## ------------------------------------------------------------------
## STEP 1: Get Data (Same)
## ------------------------------------------------------------------
data = yf.download('SPY', start='2018-01-01', end='2023-12-31')

## ------------------------------------------------------------------
## STEP 2: Feature Engineering (Same 4 features)
## ------------------------------------------------------------------
data['ROC'] = data['Close'].pct_change(10)
data['RSI'] = data['Close'].diff(1).where(data['Close'].diff(1) > 0, 0).rolling(window=14).mean() / \
              -data['Close'].diff(1).where(data['Close'].diff(1) < 0, 0).rolling(window=14).mean()
data['RSI'] = 100 - (100 / (1 + data['RSI']))
data['SMA_20'] = (data['Close'] / data['Close'].rolling(window=20).mean()) - 1
high_low = data['High'] - data['Low']
high_close = np.abs(data['High'] - data['Close'].shift())
low_close = np.abs(data['Low'] - data['Close'].shift())
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
data['ATR_14'] = tr.rolling(window=14).mean()

## ------------------------------------------------------------------
## STEP 3: Create Labels (Same 3 labels)
## ------------------------------------------------------------------
future_return = data['Close'].pct_change(5).shift(-5)
UPPER_THRESHOLD = 0.02
LOWER_THRESHOLD = -0.02
conditions = [(future_return > UPPER_THRESHOLD), (future_return < LOWER_THRESHOLD)]
choices = [1, -1]
data['Target'] = np.select(conditions, choices, default=0)

## ------------------------------------------------------------------
## STEP 4: Prepare Data for the Model (Same)
## ------------------------------------------------------------------
data = data.dropna()
feature_list = ['ROC', 'RSI', 'SMA_20', 'ATR_14']
X = data[feature_list]
Y = data['Target']

split_percentage = 0.8
split_index = int(len(data) * split_percentage)
X_train = X[:split_index]
Y_train = Y[:split_index]
X_test = X[split_index:]
Y_test = Y[split_index:]

## ------------------------------------------------------------------
## STEP 5: *** NEW - RUN RFE TO FIND THE BEST 2 FEATURES ***
## ------------------------------------------------------------------
print("Running RFE to find best features...")

# 1. Create the "engine" model that RFE will use
# This is a simple, fast model just for the ranking
estimator = RandomForestClassifier(n_estimators=50, random_state=42)

# 2. Create the RFE wrapper
# We tell it to use our estimator and to select the top 2 features
rfe = RFE(estimator=estimator, n_features_to_select=2, step=1)

# 3. Run the RFE process on our training data
rfe.fit(X_train, Y_train)

# 4. See the results
print("\n--- RFE Results ---")
print(f"Original features: {feature_list}")
print(f"Selected mask:     {rfe.support_}")
print(f"Feature ranking:   {rfe.ranking_}")

# Get the names of the winning features
selected_features = X_train.columns[rfe.support_].tolist()
print(f"Selected features: {selected_features}")

## ------------------------------------------------------------------
## STEP 6: *** NEW - PREPARE DATA WITH *ONLY* THE BEST FEATURES ***
## ------------------------------------------------------------------
# Now we filter our X data to ONLY use the features RFE selected
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

print(f"\nTraining final model with {len(selected_features)} selected features...")

## ------------------------------------------------------------------
## STEP 7: Train and Evaluate the FINAL Model
## ------------------------------------------------------------------
# We create a new, more powerful model for the final prediction
final_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    max_depth=5
)

# Train the model ONLY on the selected features
final_model.fit(X_train_selected, Y_train)

# Make predictions using the selected features
predictions = final_model.predict(X_test_selected)

print(f"Final Model Accuracy: {accuracy_score(Y_test, predictions) * 100:.2f}%")

/tmp/ipykernel_563/380623624.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('SPY', start='2018-01-01', end='2023-12-31')
[*********************100%***********************]  1 of 1 completed

Running RFE to find best features...



--- RFE Results ---
Original features: ['ROC', 'RSI', 'SMA_20', 'ATR_14']
Selected mask:     [False False  True  True]
Feature ranking:   [2 3 1 1]
Selected features: [('SMA_20', ''), ('ATR_14', '')]

Training final model with 2 selected features...
Final Model Accuracy: 68.79%


In [4]:
import yfinance as yf
import pandas_ta as ta
import pandas as pd

# 1. Get sample data from yfinance
data = yf.download("SPY", start="2024-01-01", end="2024-10-23")

print(f"--- Original Data (First 5 Rows) ---")
print(data.head())
print("\n")

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)
    
# 2. Use the .ta accessor to calculate and append indicators
# This is the easiest way to use pandas-ta.
# It automatically finds the 'Close', 'High', 'Low' columns.
data.ta.sma(length=50, append=True)  # Calculates 50-day SMA and adds a column 'SMA_50'
data.ta.rsi(length=14, append=True)  # Calculates 14-day RSI and adds a column 'RSI_14'
data.ta.bbands(length=20, append=True) # Calculates Bollinger Bands and adds 3 columns

# 3. View the results
# Note: The new indicator columns will have 'NaN' at the beginning
# This is normal, as they need a "lookback" period to be calculated.

print(f"--- Data with Indicators (First 5 Rows) ---")
print(data.head())
print("\n")

print(f"--- Data with Indicators (Last 5 Rows) ---")
print(data.tail())

/tmp/ipykernel_563/1321904829.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("SPY", start="2024-01-01", end="2024-10-23")
[*********************100%***********************]  1 of 1 completed

--- Original Data (First 5 Rows) ---
Price            Close        High         Low        Open     Volume
Ticker             SPY         SPY         SPY         SPY        SPY
Date                                                                 
2024-01-02  462.610413  463.608766  460.496290  462.130830  123623700
2024-01-03  458.832367  461.181382  458.225541  460.437516  103585900
2024-01-04  457.354462  460.956287  457.129336  458.352785   84232200
2024-01-05  457.980865  460.447327  456.522494  457.559976   86118900
2024-01-08  464.519012  464.665820  458.352813  458.480056   74879100


--- Data with Indicators (First 5 Rows) ---
Price            Close        High         Low        Open     Volume  SMA_50  \
Date                                                                            
2024-01-02  462.610413  463.608766  460.496290  462.130830  123623700     NaN   
2024-01-03  458.832367  461.181382  458.225541  460.437516  103585900     NaN   
2024-01-04  457.354462  460.95628

In [8]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE  

## ------------------------------------------------------------------
## STEP 1: Get Data (Same)
## ------------------------------------------------------------------
data = yf.download('SPY', start='2018-01-01', end='2023-12-31')

## ------------------------------------------------------------------
## STEP 2: Feature Engineering (Same 4 features)
## ------------------------------------------------------------------
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data.ta.roc(length=10, append=True)  # Rate of Change
data.ta.rsi(length=14, append=True)  # Relative Strength Index
data.ta.sma(length=20, append=True)  # Simple Moving Average
#data.ta.atr(length=14, append=True)  # Average True Range


## ------------------------------------------------------------------
## STEP 3: Create Labels (Same 3 labels)
## ------------------------------------------------------------------
future_return = data['Close'].pct_change(5).shift(-5)
UPPER_THRESHOLD = 0.02
LOWER_THRESHOLD = -0.02
conditions = [(future_return > UPPER_THRESHOLD), (future_return < LOWER_THRESHOLD)]
choices = [1, -1]
data['Target'] = np.select(conditions, choices, default=0)

## ------------------------------------------------------------------
## STEP 4: Prepare Data for the Model (Same)
## ------------------------------------------------------------------
data = data.dropna()
feature_list = ['ROC_10', 'RSI_14', 'SMA_20']
X = data[feature_list]
Y = data['Target']

split_percentage = 0.8
split_index = int(len(data) * split_percentage)
X_train = X[:split_index]
Y_train = Y[:split_index]
X_test = X[split_index:]
Y_test = Y[split_index:]

print("--- Class Distribution ---")
print(Y_train.value_counts(normalize=True))

## ------------------------------------------------------------------
## STEP 5: *** NEW - RUN RFE TO FIND THE BEST 2 FEATURES ***
## ------------------------------------------------------------------
print("Running RFE to find best features...")

# 1. Create the "engine" model that RFE will use
# This is a simple, fast model just for the ranking
estimator = RandomForestClassifier(n_estimators=50, random_state=42)

# 2. Create the RFE wrapper
# We tell it to use our estimator and to select the top 2 features
rfe = RFE(estimator=estimator, n_features_to_select=2, step=1)

# 3. Run the RFE process on our training data
rfe.fit(X_train, Y_train)

# 4. See the results
print("\n--- RFE Results ---")
print(f"Original features: {feature_list}")
print(f"Selected mask:     {rfe.support_}")
print(f"Feature ranking:   {rfe.ranking_}")

# Get the names of the winning features
selected_features = X_train.columns[rfe.support_].tolist()
print(f"Selected features: {selected_features}")

## ------------------------------------------------------------------
## STEP 6: *** NEW - PREPARE DATA WITH *ONLY* THE BEST FEATURES ***
## ------------------------------------------------------------------
# Now we filter our X data to ONLY use the features RFE selected
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

print(f"\nTraining final model with {len(selected_features)} selected features...")

## ------------------------------------------------------------------
## STEP 7: Train and Evaluate the FINAL Model
## ------------------------------------------------------------------
# We create a new, more powerful model for the final prediction
final_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    max_depth=5,
    class_weight='balanced'
)

# Train the model ONLY on the selected features
final_model.fit(X_train_selected, Y_train)

# Make predictions using the selected features
predictions = final_model.predict(X_test_selected)

print(f"Final Model Accuracy: {accuracy_score(Y_test, predictions) * 100:.2f}%")

/tmp/ipykernel_563/2213245785.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('SPY', start='2018-01-01', end='2023-12-31')
[*********************100%***********************]  1 of 1 completed


--- Class Distribution ---
Target
 0    0.651007
 1    0.187081
-1    0.161913
Name: proportion, dtype: float64
Running RFE to find best features...

--- RFE Results ---
Original features: ['ROC_10', 'RSI_14', 'SMA_20']
Selected mask:     [False  True  True]
Feature ranking:   [2 1 1]
Selected features: ['RSI_14', 'SMA_20']

Training final model with 2 selected features...
Final Model Accuracy: 45.64%
